# Dan's Comedy Clip Studio — free test run\nThis runs in a temporary Google Colab session. It does not require a paid hosting account. Upload your ~90 MB stand-up video from your phone, transcribe it, review suggested moments, then export selected vertical clips with subtitles.\n\n**Important:** Colab free runtimes are temporary and GPU availability is not guaranteed. Download your finished clips before closing the session. Your video is processed in this temporary runtime.

In [ ]:
!pip -q install faster-whisper==1.1.1 gradio==5.23.1 pandas

In [ ]:
import os, re, json, subprocess, tempfile, shutil, uuid, textwrap\nfrom pathlib import Path\nimport pandas as pd\nimport gradio as gr\nfrom faster_whisper import WhisperModel\n\nWORK = Path('/content/comedy_clip_studio')\nWORK.mkdir(exist_ok=True, parents=True)\nMODEL = None\n\ndef get_model():\n    global MODEL\n    if MODEL is None:\n        try:\n            import torch\n            gpu = torch.cuda.is_available()\n        except Exception:\n            gpu = False\n        MODEL = WhisperModel('small', device='cuda' if gpu else 'cpu', compute_type='float16' if gpu else 'int8')\n    return MODEL\n\ndef suggest_clips(segments, duration):\n    cues = re.compile(r'\\b(why|because|literally|apparently|obviously|ridiculous|imagine|sorry|fuck|shit|hell|what)\\b', re.I)\n    ranked=[]\n    for i,s in enumerate(segments):\n        score = 2*len(cues.findall(s['text'])) + min(len(s['text'])/90, 2)\n        if i+1 < len(segments):\n            gap=segments[i+1]['start']-s['end']\n            if 0.35 < gap < 2.8: score += 1.2\n        if score >= 1.5:\n            a=max(0,s['start']-9); b=min(duration,s['end']+7)\n            if b-a >= 8: ranked.append({'start':round(a,2),'end':round(b,2),'title':s['text'][:100],'score':round(score,2),'selected':False})\n    ranked.sort(key=lambda x:x['score'], reverse=True)\n    chosen=[]\n    for c in ranked:\n        if all(c['end'] <= x['start'] or c['start'] >= x['end'] for x in chosen): chosen.append(c)\n        if len(chosen)==20: break\n    chosen.sort(key=lambda x:x['start'])\n    return chosen\n\ndef transcribe(video):\n    if not video: raise gr.Error('Upload a video first.')\n    source = video if isinstance(video,str) else video.name\n    outdir=WORK/uuid.uuid4().hex; outdir.mkdir(parents=True)\n    dest=outdir/'source'+Path(source).suffix\n    shutil.copy2(source,dest)\n    model=get_model()\n    seg_iter, info=model.transcribe(str(dest), word_timestamps=False, vad_filter=True)\n    segs=[{'start':float(s.start),'end':float(s.end),'text':s.text.strip()} for s in seg_iter]\n    if not segs: raise gr.Error('No speech was detected in this video.')\n    clips=suggest_clips(segs,float(info.duration or segs[-1]['end']))\n    state={'folder':str(outdir),'source':str(dest),'segments':segs,'clips':clips}\n    df=pd.DataFrame(clips,columns=['start','end','title','score','selected'])\n    transcript='\\n'.join(f\"[{s['start']:.1f}s] {s['text']}\" for s in segs)\n    return df, transcript, state, f\"Transcribed {len(segs)} speech segments. Found {len(clips)} possible moments.\"\n\ndef srt_time(t):\n    ms=int(t*1000); h,ms=divmod(ms,3600000); m,ms=divmod(ms,60000); sec,ms=divmod(ms,1000)\n    return f'{h:02}:{m:02}:{sec:02},{ms:03}'\n\ndef export_selected(df,state):\n    if not state or not state.get('source'): raise gr.Error('Transcribe a video first.')\n    if df is None: raise gr.Error('Select at least one row.')\n    if isinstance(df,pd.DataFrame): rows=df.to_dict('records')\n    else: rows=pd.DataFrame(df).to_dict('records')\n    folder=Path(state['folder']); source=state['source']; outputs=[]\n    for idx,row in enumerate(rows,1):\n        selected=row.get('selected',False)\n        if str(selected).lower() not in ('true','1','yes'): continue\n        start=float(row['start']); end=float(row['end'])\n        if start<0 or end<=start: continue\n        srt=folder/f'clip-{idx:02}.srt'; mp4=folder/f'clip-{idx:02}.mp4'\n        entries=[]\n        for seg in state['segments']:\n            a=max(start,seg['start']); b=min(end,seg['end'])\n            if b>a: entries.append(f\"{len(entries)+1}\\n{srt_time(a-start)} --> {srt_time(b-start)}\\n{seg['text']}\\n\")\n        srt.write_text('\\n'.join(entries),encoding='utf-8')\n        sub=str(srt).replace('\\\\','/').replace(':','\\\\:')\n        vf=f\"scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,subtitles='{sub}':force_style='FontSize=18,Outline=2,Shadow=1,Alignment=2,MarginV=150'\"\n        cmd=['ffmpeg','-y','-ss',str(start),'-i',source,'-t',str(end-start),'-vf',vf,'-c:v','libx264','-preset','veryfast','-crf','23','-c:a','aac','-b:a','128k',str(mp4)]\n        p=subprocess.run(cmd,capture_output=True,text=True)\n        if p.returncode: raise gr.Error(p.stderr[-1500:])\n        outputs.append(str(mp4))\n    if not outputs: raise gr.Error('Tick the selected column for at least one clip, then export.')\n    return outputs, f'Exported {len(outputs)} clip(s). Download them below.'\n\nwith gr.Blocks(title=\"Dan's Comedy Clip Studio\",theme=gr.themes.Soft()) as demo:\n    gr.Markdown(\"# 🎤 Dan's Comedy Clip Studio\\nUpload a set, find candidate moments, edit the timings and export vertical clips.\")\n    video=gr.Video(label='Choose your stand-up video',sources=['upload'],format='mp4')\n    run=gr.Button('Transcribe & find moments',variant='primary')\n    status=gr.Markdown()\n    state=gr.State({})\n    table=gr.Dataframe(headers=['start','end','title','score','selected'],datatype=['number','number','str','number','bool'],row_count=(0,'dynamic'),col_count=(5,'fixed'),interactive=True,label='Suggested moments — tick selected and adjust start/end seconds')\n    transcript=gr.Textbox(label='Full transcript',lines=8)\n    run.click(transcribe,inputs=video,outputs=[table,transcript,state,status])\n    export=gr.Button('Export selected 9:16 clips with subtitles',variant='primary')\n    files=gr.File(label='Finished clips — download to your phone',file_count='multiple')\n    export_status=gr.Markdown()\n    export.click(export_selected,inputs=[table,state],outputs=[files,export_status])\n\ndemo.queue(default_concurrency_limit=1).launch(share=True, debug=True)

## How to use this on Android\n1. Open this notebook in Google Colab.\n2. Tap **Runtime → Run all** (or run the cells from top to bottom).\n3. Wait for the temporary Gradio link to appear near the bottom.\n4. Open that link, upload your ~90 MB video, and wait for transcription.\n5. Tick the moments you want, then export and download the clips.\n\nKeep the Colab tab/session running until downloads are complete. Free sessions can disconnect and GPU availability can vary.